## TP4 - Michel Jean Joseph Donnet

In [16]:
## imports
import numpy as np
import time
import matplotlib.pyplot as plt
import statistics

### PART 1 - Monday, October 31, 2022

In [17]:
def cities_coordinates(filename):
    '''
    read .dat file and return a matrix with first column the city names, and coordinate in rest.

    Parameters:
    -----------
    filename

    Returns:
    --------
        matrix of strings
    '''
    return np.genfromtxt(filename, dtype=str)

In [ ]:
class AT:
    def __init__(self, cities: np.ndarray, alpha: float = 5, beta: float = 1) -> None:
        '''
        Initialize class AT with a city set, and alpha and beta parameters.
        T : dictionary of pheremone quantities between points (c1, c2)
    
        Parameters:
        -----------
        cities : set of cities that contains at first column city names and in the next two columns
                both x and y coordinate
        alpha : parameter that controls the relative importance of the pheromone
        beta : parameter that controls the relative importance of the heuristic information η_ij
        '''
        self.cities = cities[:, 1:].astype(float)
        self.cities_names = cities[:, 0]
        self.num_cities = cities.shape[0]
        self.distance_matrix = self.compute_distances()
        self.Q = np.mean([self.find_greedy()[1] for _ in range(100)])
        self.T = (np.ones((self.num_cities, self.num_cities)) - np.eye(self.num_cities)) / self.Q
        self.alpha = alpha
        self.beta = beta


    # compute distances between all points and store it into a matrix
    def compute_distances(self):
        # TODO
        '''
        Compute the distances between all points of coordinates of cities and store it into a matrix
        The returned matrix is symetric because distance between city i and j is same than distance between city j and i.

        Returns:
        --------
        M : Matrix with M[i, j] the distance between city i and j.
        '''
        N = len(self.cities_names)
        M = np.zeros((N, N))
        for i in range(N):
            for j in range(N):
                M[i, j] = np.linalg.norm(self.cities[i] - self.cities[j])
        assert np.allclose(M, M.T), "Distance matrix must be symetric !"
        return M
 
    # compute fitness by computing length of the path    
    def compute_fitness(self, combination: np.ndarray):
        # TODO
        '''
        compute path length of a combination (of cities)
    
        Parameters:
        -----------
        combination : 1D array of cities' path
    
        Returns:
        --------
        fitness : path length
        '''
        return np.sum(self.distance_matrix[combination, np.roll(combination, 1)])
    
    # compute the probability of choosing a next city from unvisited cities
    def prob_to_city(self, i, j, J):
        # TODO
        '''
        compute the probability of moving from city i to city j (j belongs to J)
    
        Parameters:
        -----------
        i : current city
        j : city belonging to J 
        J : list of unvisited cities
        self.T : dictionary of pheremone quantities between points (c1, c2)
        self.alpha : parameter that controls the relative importance of the pheromone
        self.beta : parameter that controls the relative importance of the heuristic information η_ij
    
        Returns:
        --------
        p : probability to go to city j from city i
        '''
        if j not in J:
            return 0
        numerator = self.T[i, j]**self.alpha * (1 / self.distance_matrix[i, j])**self.beta
        denominator = np.sum([self.T[i, l]**self.alpha * (1 / self.distance_matrix[i, l])**self.beta for l in J])
        return numerator / denominator
    
    
    # find solution by greedy algorithm
    def find_greedy(self):
        # TODO
        '''
        perform the greedy algorithm
    
        Returns:
        --------
        c : path (list)
        fitness : fitness of path
        exec_time : execution time
        '''
        start = time.time()
        path = [np.random.choice(np.arange(self.num_cities).astype(int))]
        for i in range(self.num_cities - 1):
            distance = np.linalg.norm(self.cities - self.cities[path[i]], axis=1)
            distance[path] = np.inf
            path.append(np.argmin(distance))
        end = time.time()
        return np.array(path), self.compute_fitness(np.array(path)), end - start
    
    
    # path of each individual ant (run later in parallel, independent at each time t)
    def ant_path(self):
        # TODO
        '''
        path of each individual ant (out of m ants)
        this function can be parallelized and used later in the main AS function
    
        Returns:
        --------
        c : path (list)
        fitness : fitness of path
        '''
        cities = list(np.arange(self.num_cities))
        visited = [np.random.choice(cities)]
        cities.remove(visited[-1])
        while cities != []:
            probabilities = [self.prob_to_city(visited[-1], cities[i], cities) for i in range(len(cities))]
            visited.append(np.random.choice(cities, p=probabilities))
            cities.remove(visited[-1])
        fitness = self.compute_fitness(np.array(visited))
        Delta = np.zeros_like(self.distance_matrix)
        index = np.arange(len(visited))
        Delta[index, np.roll(index, 1)] = self.Q / fitness
        Delta[np.roll(index, 1), index] = self.Q / fitness
        assert np.allclose(Delta, Delta.T), "Matrix Delta must be symetric !"
        return visited, fitness, Delta
    
    # find solution by Ant System Algorithm
    def find_AS(self, tmax, ants, rho, alpha, beta):
        # TODO
        '''
        perform the Ant System algorithm

        Parameters:
        -----------
        setting : dictionary of cities' coordinates in form k:v
        where k is the city name
        & v are the coordinates (x, y)
        g_fitness : fitness that we get from greedy algorithm (best one)
        dist : a dictionary of distances between all points (c1, c2) in the setting
        t_max : number of iterations 
        ants : number of ants
        alpha : controls the relative importance of the pheromone
        beta : controls the relative importance of the heuristic information η_ij

        Returns:
        --------
        c : path (list)
        fitness : fitness of path
        exec_time : execution time

        NOTE : the function ant_path can be parallelized, since at each iteration, ants are independent
        '''
        start = time.time()
        self.alpha = alpha
        self.beta = beta
        best_fitness = np.inf
        best_path = None
        for i in range(tmax):
            self.T *= (1 - rho)
            for k in range(ants):
                visited, fitness, delta = self.ant_path()
                self.T += delta
                if fitness < best_fitness:
                    best_path = visited
                    best_fitness = fitness
        return best_path, best_fitness, time.time() - start
    
    # plot the path of a combination
    def plot_path(self, path, title, filename):
        '''
        plot the path of a combination 

        Parameters:
        -----------
        combination : list of cities' path
        setting : dictionary of cities' coordinates in form k:v
        where k is the city name
        & v are the coordinates (x, y)
        title : title of plot
        filename : name to save output figure (can be removed)
        '''

        # get coordinates of each city (in order)
        coords = [self.cities[city] for city in path]
        fig = plt.gcf()
        fig.set_size_inches(15, 15)
        x,y = list(zip(*coords))
        plt.scatter(x, y, c=['r'] + ['k'] * (len(path) - 1))
        plt.title(f'{title}-{self.num_cities}cities')
        plt.gca().set_aspect('equal', adjustable='box')

        data_xrange = plt.xlim()[1] - plt.xlim()[0]
        head_scale = data_xrange * 0.3

        for i in range(len(coords)):
            x,y = list(zip(*(coords[i], coords[(i + 1) % len(coords)])))
            plt.arrow(x[0], y[0], (x[1] - x[0]), (y[1] - y[0]), length_includes_head=True,
              head_width=0.05 * head_scale, head_length=0.05 * head_scale, overhang=0.5, alpha=0.6)
        plt.axis("off")
        plt.savefig(filename)
        plt.show()

    # plot box_plot for statistics    
    def box_plot(self, methods, results):
        '''
        plot statistics' box plot

        Parameters:
        -----------
        setting : dictionary of cities' coordinates in form k:v
        where k is the city name
        & v are the coordinates (x, y)
        methods : list of methods ['greedy', 'AS']
        results : path, fitness, and execution time results of each method
        '''
        # for every setting, for every method, plot fitness boxplot
        boxes = {m: results[m]['fitness'] for m in methods}
        fig, ax = plt.subplots()
        ax.boxplot(boxes.values(), showmeans = True, labels=list(boxes.keys()))

        plt.title(f'Fitness Distribution for {self.num_cities} cities')
        plt.savefig(f'Fitness Distribution for {self.num_cities} cities')
        plt.show(fig)
        plt.close(fig)

    def compare_methods(self, tmax, ants, alpha, beta, filename=None):
        # TODO (Some Parts)
        '''
        compare greedy & AS methods, output statistics to file + box_plot, & plot paths for initial, best greedy, and best AS 

        Parameters:
        -----------
        setting : dictionary of cities' coordinates in form k:v
        where k is the city name
        & v are the coordinates (x, y)
        t_max : number of iterations 
        ants : number of ants
        alpha : controls the relative importance of the pheromone
        beta : controls the relative importance of the heuristic information η_ij

        Returns:
        --------
        c : path (list)
        fitness : fitness of path
        exec_time : execution time

        NOTE : the function ant_path can be parallelized, since at each iteration, ants are independent
        '''
        methods = ('greedy', 'AS')
        results = dict.fromkeys(methods)
        for m in methods:
            results[m] = {
                'path': [],
                'fitness': [],
                'exec_time': []
            }

        #run greedy 10 times
        for _ in range(10):
            path, fitness, exec_time = self.find_greedy()
            results['greedy']['path'].append(path)
            results['greedy']['fitness'].append(fitness)
            results['greedy']['exec_time'].append(exec_time)

        ## TODO --------------------------------
        # get g_fitness (L_nn), best solution of the greedy algorithm
        self.Q = np.min(results['greedy']['fitness'])
        # --------------------------------------

        #run AS 5 times
        for _ in range(5):
            path, fitness, exec_time = self.find_AS(tmax, ants, alpha, beta)
            results['AS']['path'].append(path)
            results['AS']['fitness'].append(fitness)
            results['AS']['exec_time'].append(exec_time)

        self.box_plot( methods, results)

        # for every method, print statistics 
        min_greedy_fitness = min(results['greedy']['fitness'])
        std_greedy_fitness = statistics.stdev(results['greedy']['fitness'])
        mean_greedy_fitness = statistics.mean(results['greedy']['fitness'])
        std_greedy_exec = statistics.stdev(results['greedy']['exec_time'])
        mean_greedy_exec = statistics.mean(results['greedy']['exec_time'])
        min_greedy_path = results['greedy']['path'][results['greedy']['fitness'].index(min_greedy_fitness)]
        min_as_fitness = min(results['AS']['fitness'])
        std_as_fitness = statistics.stdev(results['AS']['fitness'])
        mean_as_fitness = statistics.mean(results['AS']['fitness'])
        std_as_exec = statistics.stdev(results['AS']['exec_time'])
        mean_as_exec = statistics.mean(results['AS']['exec_time'])
        min_as_path = results['AS']['path'][results['AS']['fitness'].index(min_as_fitness)]

        outfile = "Results"
        if filename:
            outfile += "_" + filename
        outfile += ".txt"
        f = open(outfile, "a")
        s0 = f'\n For {self.num_cities} cities:\n'
        s1 = f'Greedy: Best Path = {min_greedy_path}, Minimum Fitness = {min_greedy_fitness}, Mean Fitness= {mean_greedy_fitness}, Std Fitness= {std_greedy_fitness}, Mean exec_time= {mean_greedy_exec}, Std exec_time= {std_greedy_exec}\n'
        s2 = f'AS: Best Path = {min_as_path}, Minimum Fitness = {min_as_fitness}, Mean Fitness= {mean_as_fitness}, Std Fitness= {std_as_fitness}, Mean exec_time= {mean_as_exec}, Std exec_time= {std_as_exec}\n'
        f.write(s0+s1+s2)
        f.close()

        ## TODO --------------------------------
        # choose initial combination 
        initial = np.arange(self.num_cities)
        np.random.shuffle(initial)
        # shuffle initial
        # --------------------------------------

        self.plot_path(initial, 'Initial', f"path_initial_{self.num_cities}.png")
        self.plot_path(min_greedy_path, 'Greedy', f"path_greedy_{self.num_cities}.png")
        self.plot_path(min_as_path, 'AS', f"path_AS_{self.num_cities}_{tmax}_{ants}_{alpha}_{beta}.png")


    def compare_AS_tmax(self, g_fitness, ants, alpha, beta):
        # TODO (Some Parts)
        '''
        compare AS algorithm for different tmax & plot

        Parameters:
        -----------
        setting : dictionary of cities' coordinates in form k:v
        where k is the city name
        & v are the coordinates (x, y)
        dist : a dictionary of distances between all points (c1, c2) in the setting
        g_fitness : fitness that we get from greedy algorithm (best one)
        ants : number of ants
        alpha : controls the relative importance of the pheromone
        beta : controls the relative importance of the heuristic information η_ij
        '''
        ## TODO --------------------------------
        # different possibilities of tmax
        tmax = self.num_cities**3
        # --------------------------------------
        self.Q = g_fitness
        all_c = []
        all_fitness = []
        all_exec_time = []
        for t in tmax:
            c, fitness, exec_time = self.find_AS(tmax=t, ants=ants, rho=0.5, alpha=alpha, beta=beta)
            all_fitness.append(fitness)
            all_exec_time.append(exec_time)
        plt.plot(tmax, all_fitness, c='b')
        plt.xlabel("Tmax")
        plt.ylabel("Fitness")
        plt.savefig("Fitness-Iterations.png")
        plt.show()
        plt.plot(tmax, all_exec_time, c='r')
        plt.xlabel("Tmax")
        plt.ylabel("Execution Time")
        plt.savefig("Time-Tmax.png")
        plt.show()

    def compare_AS_ants(self, g_fitness, tmax, alpha, beta):
        '''
        compare AS algorithm for different tmax & plot

        Parameters:
        -----------
        setting : dictionary of cities' coordinates in form k:v
        where k is the city name
        & v are the coordinates (x, y)
        dist : a dictionary of distances between all points (c1, c2) in the setting
        g_fitness : fitness that we get from greedy algorithm (best one)
        t_max : number of iterations 
        alpha : controls the relative importance of the pheromone
        beta : controls the relative importance of the heuristic information η_ij
        '''
        ## TODO --------------------------------
        # different possibilities of ants
        ants = self.num_cities
        # --------------------------------------
        self.Q = g_fitness
        all_c = []
        all_fitness = []
        all_exec_time = []
        for m in ants:
            c, fitness, exec_time = self.find_AS(tmax=tmax, ants=m, alpha=alpha, beta=beta)
            all_fitness.append(fitness)
            all_exec_time.append(exec_time)
        plt.plot(ants, all_fitness, c='b')
        plt.xlabel("Number of ants")
        plt.ylabel("Fitness")
        plt.savefig("Fitness-Ants.png")
        plt.show()
        plt.plot(ants, all_exec_time, c='r')
        plt.xlabel("Number of ants")
        plt.ylabel("Execution Time")
        plt.savefig("Time-Ants.png")
        plt.show()      




In [14]:
data = cities_coordinates("../data/cities.dat")
test = AT(data)
results = test.find_greedy()
print(f"Greedy path: {results[0]}")
print(f"Greedy fitness: {results[1]}")
print(f"Greedy time: {results[2]}")
print(test.ant_path()[:2])

Greedy path: [ 0  1 13  9  3 11 16  8  5  7  2  4 10 14 17  6 15 12]
Greedy fitness: 36.16128455775342
Greedy time: 0.00010514259338378906
([np.int64(3), np.int64(5), np.int64(10), np.int64(17), np.int64(11), np.int64(0), np.int64(1), np.int64(8), np.int64(6), np.int64(2), np.int64(13), np.int64(4), np.int64(16), np.int64(9), np.int64(7), np.int64(14), np.int64(15), np.int64(12)], np.float64(57.14946279903448))


##### NOTE: Completing the AS algorithm (function find_AS) is included in Part 2 of this TP (due Nov 7th), thus will not be part of the evaluation for Part 1. However, it is highly advised to try to work on it during TP4 Part1 if time permits, so that we can give feedback to aid for Part 2. 

### PART 2 - Monday, November 7, 2022

In [8]:
def compare_AS_tmax(setting, dist, g_fitness, ants=??, alpha=??, beta=??):
    # TODO (Some Parts)
    '''
    compare AS algorithm for different tmax & plot

    Parameters:
    -----------
    setting : dictionary of cities' coordinates in form k:v
    where k is the city name
    & v are the coordinates (x, y)
    dist : a dictionary of distances between all points (c1, c2) in the setting
    g_fitness : fitness that we get from greedy algorithm (best one)
    ants : number of ants
    alpha : controls the relative importance of the pheromone
    beta : controls the relative importance of the heuristic information η_ij
    '''
    ## TODO --------------------------------
    # different possibilities of tmax
    tmax = [??]
    # --------------------------------------
    all_c = []
    all_fitness = []
    all_exec_time = []
    for t in tmax:
        c, fitness, exec_time = find_AS(setting, g_fitness, dist, tmax=t, ants=ants, alpha=alpha, beta=beta)
        all_fitness.append(fitness)
        all_exec_time.append(exec_time)
    plt.plot(tmax, all_fitness, c='b')
    plt.xlabel("Tmax")
    plt.ylabel("Fitness")
    plt.savefig("Fitness-Iterations.png")
    plt.show()
    plt.plot(tmax, all_exec_time, c='r')
    plt.xlabel("Tmax")
    plt.ylabel("Execution Time")
    plt.savefig("Time-Tmax.png")
    plt.show()

def compare_AS_ants(setting, dist, g_fitness, tmax=??, alpha=??, beta=??):
    '''
    compare AS algorithm for different tmax & plot

    Parameters:
    -----------
    setting : dictionary of cities' coordinates in form k:v
    where k is the city name
    & v are the coordinates (x, y)
    dist : a dictionary of distances between all points (c1, c2) in the setting
    g_fitness : fitness that we get from greedy algorithm (best one)
    t_max : number of iterations 
    alpha : controls the relative importance of the pheromone
    beta : controls the relative importance of the heuristic information η_ij
    '''
    ## TODO --------------------------------
    # different possibilities of ants
    ants = [??]
    # --------------------------------------

    all_c = []
    all_fitness = []
    all_exec_time = []
    for m in ants:
        c, fitness, exec_time = find_AS(setting, g_fitness, dist, tmax=tmax, ants=m, alpha=alpha, beta=beta)
        all_fitness.append(fitness)
        all_exec_time.append(exec_time)
    plt.plot(ants, all_fitness, c='b')
    plt.xlabel("Number of ants")
    plt.ylabel("Fitness")
    plt.savefig("Fitness-Ants.png")
    plt.show()
    plt.plot(ants, all_exec_time, c='r')
    plt.xlabel("Number of ants")
    plt.ylabel("Execution Time")
    plt.savefig("Time-Ants.png")
    plt.show()

SyntaxError: invalid syntax (2257411878.py, line 1)

In [ ]:
# dictionary of coordinates of cities from 'cities.dat' file
d_c1 = cities_coordinates('cities.dat')
# dictionary of coordinates of cities from 'cities2.dat' file
d_c2 = cities_coordinates('cities2.dat')

In [ ]:
# dictionary of dictionaries for each 50, 60, 80, and 100 cities with their coordinates
c = [50, 60, 80, 100]
d_pt = OrderedDict()
for i in c:
    points = [str("p"+str(x)) for x in range(i)]
    coord = [(random.uniform(-10, 10), random.uniform(-10, 10)) for _ in range(i)]
    value = zip(points, coord)

    d_pt[i] = d50 = {k: v for (k, v) in value}

In [ ]:
## Study effect of parameters, example on 'cities2.dat' file
dist = compute_distances(d_c2)
#run greedy once to get Lnn
path, g_fitness, exec_time = find_greedy(d_c2, dist)
# Tmax variation
compare_AS_tmax(d_c2, dist, g_fitness)
# Number of Ants variation
compare_AS_ants(d_c2, dist, g_fitness)

#### Comment on the results (effect of t_max & number of ants)

In [ ]:
## compare greedy vs AS for 'cities.dat', 'cities2.dat', and 50, 60, 80, & 100 cities' configurations
compare_methods(setting=d_c1, filename="c1")
compare_methods(setting=d_c2, filename="c2")
for k in d_pt:
    compare_methods(setting=d_pt[k], filename=f'dpt_{k}')

#### Comment on the results for Ant System (AS) vs. Greedy

#### Compare and discuss the performance of the Ant System (AS) and the Simulated Annealing (SA) algorithms applied on the TSP in terms of their solution quality and execution time.

Answer the following Questions: 

1.  Q: What is the Search Space for Ant System algorithm? (given number of cities = n, & number of ants = m)

    A: 


2.  Q: Can you describe the neighborhood in this case?

    A:

3.  Q: Talk about the impact of parameters $\alpha$, $\beta$, & $\rho$

    A:
